# Gemma 3 + torch 2.5.1 검증 (Tier C)

과제 명세 지정 환경 **torch 2.5.1**에서 폴백 모델 **Gemma 3**가 도는지 확인.
Gemma 4는 torch 2.7+(float8_e8m0fnu) 필요라 2.5.1에서 막혔지만, Gemma 3는 그 dtype을
안 써서 가능. 단 **transformers를 4.50대로 내려야** torch 2.5.1과 짝이 맞음.

## 실행 순서 (Colab T4, 새 세션)
1. **[1단계]** torch 2.5.1 설치 → 자동 재시작
2. **[2단계]** transformers 4.50 + bitsandbytes 설치(torch 핀 고정) → 자동 재시작
3. **[3단계]** 버전/임포트 판정
4. **[HF 로그인]** (gemma-3는 gated)
5. **[4단계]** 모델 로드 + 생성 + RAG 테스트

> 빨간 에러가 떠도 그것도 결과다 — 어디서 막히는지가 정보.

## [1단계] torch 2.5.1 설치 → 자동 재시작

In [ ]:
import os
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1
print("torch 2.5.1 설치 완료 — 재시작합니다. 재시작되면 [2단계]부터.")
os.kill(os.getpid(), 9)

## [2단계] (재시작 후) transformers 4.50 + bitsandbytes 설치 → 자동 재시작
torch를 2.5.1로 핀에 박아 pip가 다시 올리지 못하게 한다.

In [ ]:
import torch, os
print("현재 torch:", torch.__version__)
assert torch.__version__.startswith("2.5.1"), "torch가 2.5.1이 아님 — [1단계]부터 다시"

!pip install -q transformers==4.50.0 accelerate "bitsandbytes>=0.45" \
    torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1

print("\n설치 완료 — 재시작합니다. 재시작되면 [3단계]부터.")
os.kill(os.getpid(), 9)

## [3단계] (재시작 후) 판정 — torch 2.5.1 유지 + Gemma 3 임포트

In [ ]:
import importlib.metadata as md
import torch

print("torch:", md.version("torch"), "| cuda:", torch.cuda.is_available())
print("transformers:", md.version("transformers"))

ok, err = False, ""
try:
    from transformers import AutoModelForImageTextToText  # gemma3 매핑
    ok = True
except Exception as e:
    err = f"{type(e).__name__}: {e}"

print("Gemma3 임포트 가능:", ok)
if not ok:
    print("  └", err)
print("-" * 50)
if md.version("torch").startswith("2.5.1") and ok:
    print("판정 OK : torch 2.5.1에서 Gemma 3 임포트 성공 → [4단계] 진행")
else:
    print("판정 보류 : 위 에러/버전 확인")

## [HF 로그인] (gemma-3는 gated — 라이선스 먼저 수락)

In [ ]:
# from huggingface_hub import login
# login()

## [4단계] Gemma 3 로드 + 생성 + RAG 테스트

In [ ]:
import os
os.environ.setdefault("HF_HOME", "/content/hf_cache")

import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_ID = "google/gemma-3-12b-it"   # 더 가볍게: "google/gemma-3-4b-it"
COMPUTE_DTYPE = torch.float16        # T4는 bf16 텐서코어 없음

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

print("프로세서/모델 로드 중...")
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto", torch_dtype=COMPUTE_DTYPE,
)
model.eval()
print(f"로드 완료 — VRAM {torch.cuda.memory_reserved()/1e9:.2f} GB | torch {torch.__version__}")

def gen(text, max_new_tokens=256):
    msgs = [{"role": "user", "content": [{"type": "text", "text": text}]}]
    inputs = processor.apply_chat_template(
        msgs, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)
    n = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.3)
    return processor.decode(out[0][n:], skip_special_tokens=True).strip()

print("\n--- 생성 테스트 ---")
print(gen("충남대학교에 대해 한 문장으로 설명해줘."))

print("\n--- RAG 스타일 테스트 ---")
ctx = "컴퓨터융합학부 졸업요건: 졸업에 필요한 총 학점은 130학점. 교양 최대 48학점 인정. 프로젝트 교과목 최소 3개 이수."
q = "컴퓨터융합학부 졸업하려면 몇 학점이야?"
print(gen(f"너는 충남대 안내 챗봇이야. 참고 자료에 있는 내용만으로 답해.\n\n참고 자료:\n{ctx}\n\n질문: {q}", 300))